In [1]:
# ============================================================
# CAPSTONE PROJECT: Malicious URL Detector
# ============================================================

import pandas as pd
import numpy as np
import re
import joblib
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ------------------------------------------------------------
# STEP 1: Dataset load karo (651K labeled URLs)
# ------------------------------------------------------------
url_source = 'https://raw.githubusercontent.com/mango-cat/ECS171-Project/main/malicious_phish.csv'
df = pd.read_csv(url_source)
df['label'] = df['type'].apply(lambda x: 0 if x == 'benign' else 1)

# Balanced sample lo (50,000 URLs) speed ke liye
benign_sample = df[df['label'] == 0].sample(n=25000, random_state=42)
malicious_sample = df[df['label'] == 1].sample(n=25000, random_state=42)
df_sample = pd.concat([benign_sample, malicious_sample]).reset_index(drop=True)

# ------------------------------------------------------------
# STEP 2: URL cleaning function (protocol/www hata do - data leakage se bachne ke liye)
# ------------------------------------------------------------
def clean_url(u):
    u = u.replace('https://', '').replace('http://', '')
    if u.startswith('www.'):
        u = u[4:]
    return u

df_sample['url_clean'] = df_sample['url'].apply(clean_url)

# ------------------------------------------------------------
# STEP 3: Popular safe domains ko augment karo (dataset mein bare domains kam thay)
# ------------------------------------------------------------
safe_domains = ['google.com', 'facebook.com', 'youtube.com', 'amazon.com', 'wikipedia.org',
                 'twitter.com', 'instagram.com', 'linkedin.com', 'microsoft.com', 'apple.com',
                 'netflix.com', 'reddit.com', 'yahoo.com', 'ebay.com', 'github.com',
                 'stackoverflow.com', 'whatsapp.com', 'zoom.us', 'dropbox.com', 'spotify.com',
                 'pinterest.com', 'tumblr.com', 'quora.com', 'bbc.com', 'cnn.com',
                 'nytimes.com', 'espn.com', 'imdb.com', 'paypal.com', 'adobe.com']

extra_urls = []
for domain in safe_domains:
    extra_urls.append(domain)
    extra_urls.append('www.' + domain)
    extra_urls.append(domain + '/')

extra_df = pd.DataFrame({'url_clean': [clean_url(u) for u in extra_urls], 'label': 0})
extra_df_boosted = pd.concat([extra_df] * 20, ignore_index=True)  # weight badhane ke liye 20x duplicate

combined_df = pd.concat([
    df_sample[['url_clean', 'label']],
    extra_df_boosted
], ignore_index=True)

# ------------------------------------------------------------
# STEP 4: Train/test split
# ------------------------------------------------------------
X_text = combined_df['url_clean']
y = combined_df['label']

X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42, stratify=y)

# ------------------------------------------------------------
# STEP 5: TF-IDF (character n-grams) + Logistic Regression pipeline
# ------------------------------------------------------------
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=8000)),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

model_pipeline.fit(X_train, y_train)
pred = model_pipeline.predict(X_test)

# ------------------------------------------------------------
# STEP 6: Evaluate
# ------------------------------------------------------------
print("=== Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, pred):.2%}")
print(f"Precision: {precision_score(y_test, pred):.2%}")
print(f"Recall:    {recall_score(y_test, pred):.2%}")
print(f"F1 Score:  {f1_score(y_test, pred):.2%}")

# ------------------------------------------------------------
# STEP 7: Real-world sanity check
# ------------------------------------------------------------
print("\n=== Real-World URL Tests ===")
test_urls = ["google.com", "facebook.com", "amazon.com", "youtube.com", "wikipedia.org",
             "paypa1-secure-login.com/account/verify", "bit.ly/xyz123abc",
             "192.168.1.1/admin/login.php", "amaz0n-account-verify.tk"]

for test_url in test_urls:
    tu = clean_url(test_url)
    p = model_pipeline.predict([tu])[0]
    r = model_pipeline.predict_proba([tu])[0][1]
    label = "MALICIOUS" if p == 1 else "SAFE"
    print(f"{test_url:45s} -> {label:10s} (risk: {r:.1%})")

# ------------------------------------------------------------
# STEP 8: Save the final model
# ------------------------------------------------------------
joblib.dump(model_pipeline, 'phishing_model.pkl')
print("\n✅ Saved: phishing_model.pkl")

from google.colab import files
files.download('phishing_model.pkl')

=== Model Performance ===
Accuracy:  87.65%
Precision: 88.02%
Recall:    86.14%
F1 Score:  87.07%

=== Real-World URL Tests ===
google.com                                    -> SAFE       (risk: 29.4%)
facebook.com                                  -> SAFE       (risk: 6.5%)
amazon.com                                    -> SAFE       (risk: 13.8%)
youtube.com                                   -> SAFE       (risk: 5.8%)
wikipedia.org                                 -> SAFE       (risk: 12.2%)
paypa1-secure-login.com/account/verify        -> SAFE       (risk: 34.4%)
bit.ly/xyz123abc                              -> MALICIOUS  (risk: 69.5%)
192.168.1.1/admin/login.php                   -> MALICIOUS  (risk: 98.2%)
amaz0n-account-verify.tk                      -> MALICIOUS  (risk: 72.5%)

✅ Saved: phishing_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>